[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/14xp/iliad-comp-mech-materials/blob/main/exercises/colab/part2_belief_states_exercises_colab.ipynb)

> **Running on Colab.** This notebook is self-contained: the **Setup** cells below write the helper modules (`processes`, `solutions`, `tests`, `plotting`) into the session, then import them — nothing else to download. Just run the cells top to bottom. (To open it: *File → Upload notebook*, or from Google Drive / a GitHub link.)

# Part 2: Belief States

After observing a sequence $x_{1:t}$, the observer's optimal summary of the past is the **belief state**: the posterior distribution over hidden states,
$$
\eta^{(x_{1:t})}_j \;=\; P\big(s_t = j \mid x_{1:t}\big), \qquad \eta^{(x_{1:t})} \in \Delta^{n_\text{states}-1}.
$$
This is a vector of shape `(n_states,)` lying on the probability simplex. The unnormalized propagated vector from Part 1, $v = \text{initial\_vect}\cdot T[x_1]\cdots T[x_t]$, already has entries $v_j = P(x_{1:t}, s_t = j)$; normalizing by $\sum_j v_j = P(x_{1:t})$ turns the joint into the posterior. So the belief state is just a *renormalized* propagated vector — but that renormalization is exactly what makes it the right state for prediction.

In this part the non-belief machinery (`_propagate`, `sequence_probability`, `conditional_next_token_probabilities`, `all_next_token_probabilities`) is **provided pre-implemented** in the HMM skeleton; you may call it freely. You will implement, in order:

1. `belief_state` — the posterior distribution $\eta^{(x_{1:t})}$ over hidden states given the observed sequence $x_{1:t}$.
2. `belief_update` — the recursive one-symbol Bayesian update $\eta^{(x_{1:t})} \mapsto \eta^{(x_{1:t+1})}$, avoiding re-propagation from scratch.
3. `ntp_from_belief_state` — the next-token distribution implied by a belief state.
4. `all_belief_states` — every reachable belief state up to a given depth, the data needed to draw the mixed-state simplex.

The workflow is identical to Part 1: write the method, monkeypatch it onto your `HMM` class, and check it with `tests.test_<method>(HMM)`.

## Setup

These cells write the helper modules into the Colab session and import them. Run them once, top to bottom.

In [ ]:
%%writefile processes.py
"""Hidden-Markov processes used as fixtures/demos in the HMM exercises.

Each process is given as an observation-indexed transition tensor of shape
(n_obs, n_states, n_states): ``tensor[x, i, j]`` is the probability of emitting
symbol ``x`` and moving from hidden state ``i`` to hidden state ``j``. Summing
over the symbol axis gives a row-stochastic state-transition matrix.
"""

import numpy as np


def arch(a: float) -> np.ndarray:
    """Creates a transition matrix for the Arch Process."""
    assert 0 <= a <= 1, f"a must be in [0, 1], got {a}"
    b = (1 - a) / 3
    return np.array(
        [
            [
                [0.8 * a, 0.0, 0.0, 0.0],
                [0.0, 0.2 * a, 0.0, 0.0],
                [0.0, 0.0, 0.4 * a, 0.0],
                [0.0, 0.0, 0.0, 0.6 * a],
            ],
            [
                [0.0, 0.0, 0.0, 0.0],
                [0.0, 0.4 * a, 0.0, 0.4 * b],
                [0.0, 0.0, 0.3 * a, 0.0],
                [0.0, 0.0, 0.0, 0.16 * a],
            ],
            [
                [0.2 * a, b, b, b],
                [b, 0.4 * a, b, 0.6 * b],
                [b, b, 0.3 * a, b],
                [b, b, b, 0.24 * a],
            ],
        ]
    )

def z1r(p: float = 0.5) -> np.ndarray:
    """Zero-One-Random (Z1R) process: 3 hidden states, 2 symbols.

    Emits 0 then 1 deterministically, then a random bit with P(1) = ``p``.
    Stationary distribution is uniform, [1/3, 1/3, 1/3].
    """
    q = 1 - p
    return np.array(
        [
            [[0, 1, 0], [0, 0, 0], [q, 0, 0]],
            [[0, 0, 0], [0, 0, 1], [p, 0, 0]],
        ],
        dtype=float,
    )


def mess3(x: float = 0.15, a: float = 0.2) -> np.ndarray:
    """Mess3 process: 3 hidden states, 3 symbols.

    A symmetric process whose belief states trace out a fractal in the
    2-simplex. Stationary distribution is uniform, [1/3, 1/3, 1/3].
    """
    b = (1 - a) / 2
    y = 1 - 2 * x
    ay, bx, by, ax = a * y, b * x, b * y, a * x
    return np.array(
        [
            [[ay, bx, bx], [ax, by, bx], [ax, bx, by]],
            [[by, ax, bx], [bx, ay, bx], [bx, ax, by]],
            [[by, bx, ax], [bx, by, ax], [bx, bx, ay]],
        ],
        dtype=float,
    )


def uniform_initial(n_states: int) -> np.ndarray:
    """Uniform initial state distribution over ``n_states`` hidden states."""
    return np.ones(n_states) / n_states

In [ ]:
%%writefile solutions.py
"""Reference HMM implementation for the exercises.

The exercise notebooks ask the reader to
reconstruct these methods; the tests in ``tests.py`` compare the reader's work
against this reference.
"""

import numpy as np
from collections.abc import Sequence

type TransitionTensor = np.ndarray  # (n_obs, n_states, n_states)
type StateVector = np.ndarray       # (n_states,) -- belief / propagated state vector
type TokenDist = np.ndarray         # (n_obs,)    -- distribution over next tokens


class HMM:
    """HMM defined by an observation-indexed transition tensor and an initial vector.

    Shapes
    ------
    transition_tensor        : (n_obs, n_states, n_states)
    belief / state vectors   : (n_states,)
    next-token distributions : (n_obs,)
    """

    def __init__(self, transition_tensor: TransitionTensor, initial_vect: StateVector) -> None:
        self.transition_tensor = np.asarray(transition_tensor)
        self.initial_vect = np.atleast_1d(np.asarray(initial_vect)).ravel()

    def validate(self) -> None:
        """Assert that transition_tensor and initial_vect define a valid HMM.

        The observation-summed state-transition matrix must be row-stochastic and
        the initial vector must be a probability distribution. Raises AssertionError
        otherwise.
        """
        assert (self.transition_tensor >= 0).all(), "transition_tensor must be non-negative"
        transition_matrix = self.transition_tensor.sum(axis=0)
        assert np.allclose(transition_matrix.sum(axis=1), 1.0), \
            "summed transition matrix must be row-stochastic (each row sums to 1)"
        assert (self.initial_vect >= 0).all(), "initial_vect must be non-negative"
        assert np.isclose(self.initial_vect.sum(), 1.0), "initial_vect must sum to 1"

    def _propagate(self, sequence: Sequence[int], vect: StateVector | None = None) -> StateVector:
        """Run a sequence through the tensor, returning the unnormalized state vector."""
        vect = self.initial_vect if vect is None else vect
        for obs in sequence:
            vect = np.einsum("i,ij->j", vect, self.transition_tensor[obs])
        return vect

    def sequence_probability(self, sequence: Sequence[int]) -> float:
        return np.sum(self._propagate(sequence))

    def conditional_next_token_probabilities(self, sequence: Sequence[int]) -> TokenDist:
        seq = list(sequence)
        p_seq = self.sequence_probability(seq)
        if np.allclose(p_seq, 0.0):
            raise ValueError(f"sequence {tuple(sequence)} has zero probability; "
                             "next-token distribution is undefined.")
        n_obs = self.transition_tensor.shape[0]
        joint = np.array([self.sequence_probability(seq + [k]) for k in range(n_obs)])
        return joint / p_seq

    def all_next_token_probabilities(self, max_depth: int) -> dict[tuple[int, ...], TokenDist]:
        """Next-token distribution for every reachable sequence of length 0..max_depth.

        Built breadth-first; a continuation is reachable iff its probability in the
        current distribution is nonzero (zero-probability branches are pruned).
        Uses conditional_next_token_probabilities directly -- no belief-state machinery.
        Includes () -> the prior next-token distribution.
        """
        prior = self.conditional_next_token_probabilities(())
        result = {(): prior}
        frontier = [((), prior)]
        for _ in range(max_depth):
            next_frontier = []
            for seq, ntp in frontier:
                for obs in range(len(ntp)):
                    if np.allclose(ntp[obs], 0.0):
                        continue  # zero-probability continuation -> prune
                    new_seq = seq + (obs,)
                    child = self.conditional_next_token_probabilities(new_seq)
                    result[new_seq] = child
                    next_frontier.append((new_seq, child))
            frontier = next_frontier
        return result

    #######################
    # Belief State Methods #
    #######################

    def belief_state(self, sequence: Sequence[int]) -> StateVector:
        vect = self._propagate(sequence)
        norm = np.sum(vect)
        if np.allclose(norm, 0.0):
            raise ValueError(f"sequence {tuple(sequence)} has zero probability; belief state is undefined.")
        return vect / norm

    def belief_update(self, belief_state: StateVector, obs: int) -> StateVector:
        update = np.einsum("i,ij->j", belief_state, self.transition_tensor[obs])
        norm = np.sum(update)
        if np.allclose(norm, 0.0):
            raise ValueError(f"observation {obs} has zero probability from this belief state; cannot update.")
        return update / norm

    def ntp_from_belief_state(self, belief_state: StateVector) -> TokenDist:
        return np.einsum("i,kij->k", belief_state, self.transition_tensor)

    def all_belief_states(self, max_depth: int) -> dict[tuple[int, ...], StateVector]:
        """Belief state for every reachable sequence of length 0..max_depth.

        Returns a dict mapping each observation sequence (as a tuple) to its
        belief state, starting from the empty sequence () -> prior belief.
        Built breadth-first via belief_update; zero-probability continuations
        are pruned (they have no defined belief state).
        """
        n_obs = self.transition_tensor.shape[0]
        prior = self.belief_state(())
        result = {(): prior}
        frontier = [((), prior)]
        for _ in range(max_depth):
            next_frontier = []
            for seq, belief in frontier:
                for obs in range(n_obs):
                    try:
                        updated = self.belief_update(belief, obs)
                    except ValueError:
                        continue  # zero-probability continuation -> prune
                    new_seq = seq + (obs,)
                    result[new_seq] = updated
                    next_frontier.append((new_seq, updated))
            frontier = next_frontier
        return result

In [ ]:
%%writefile tests.py
"""ARENA-style tests for the HMM exercises.

Each ``test_<method>(HMM)`` takes the reader's ``HMM`` class (with the relevant
method monkeypatched on) and checks it against the reference implementation in
``solutions.py``, across the Z1R and Mess3 processes. Inputs to the
method-under-test are always built from the *reference*, so each test isolates a
single method. Prints "All tests passed!" on success; raises ``AssertionError``
(or surfaces an unexpected exception) otherwise.
"""

from itertools import product

import numpy as np

import processes
import solutions


def _fixtures():
    """(label, transition_tensor, initial_vect) cases covering 2- and 3-symbol processes."""
    return [
        ("Z1R", processes.z1r(0.5), processes.uniform_initial(3)),
        ("Mess3", processes.mess3(0.15, 0.2), processes.uniform_initial(3)),
    ]


def _all_seqs(n_obs, max_len=4):
    seqs = [()]
    for length in range(1, max_len + 1):
        seqs.extend(product(range(n_obs), repeat=length))
    return seqs


def _reachable_seqs(ref, n_obs, max_len=4):
    return [s for s in _all_seqs(n_obs, max_len)
            if ref.sequence_probability(list(s)) > 1e-12]


def _first_zero_prob_seq(ref, n_obs, max_len=4):
    for s in _all_seqs(n_obs, max_len):
        if s and np.isclose(ref.sequence_probability(list(s)), 0.0):
            return s
    raise RuntimeError("no zero-probability sequence found for this fixture")


# ----------------------------------------------------------------------------
# Part 1: sequence probabilities / next-token distributions
# ----------------------------------------------------------------------------

def test_validate(HMM):
    # well-formed processes must pass
    for label, tensor, init in _fixtures():
        HMM(tensor, init).validate()
    # each malformed case must raise AssertionError
    good_t, good_i = processes.z1r(0.5), processes.uniform_initial(3)
    neg_t = good_t.copy()
    neg_t[0, 2, 0] = -neg_t[0, 2, 0] - 0.1
    bad_cases = [
        ("non-row-stochastic transition matrix", good_t * 2.0, good_i),
        ("initial_vect that does not sum to 1", good_t, np.ones(3)),
        ("negative initial_vect", good_t, np.array([1.5, -0.5, 0.0])),
        ("negative transition_tensor entry", neg_t, good_i),
    ]
    for desc, tensor, init in bad_cases:
        try:
            HMM(tensor, init).validate()
        except AssertionError:
            pass
        else:
            raise AssertionError(f"validate() should have rejected: {desc}")
    print("All tests passed!")


def test_propagate(HMM):
    for label, tensor, init in _fixtures():
        ref, cand = solutions.HMM(tensor, init), HMM(tensor, init)
        n_obs = tensor.shape[0]
        for s in _all_seqs(n_obs):
            expected = ref._propagate(list(s))
            actual = cand._propagate(list(s))
            assert np.shape(actual) == np.shape(expected), (
                f"[{label}] _propagate{tuple(s)} has shape {np.shape(actual)}, "
                f"expected {np.shape(expected)}")
            assert np.allclose(actual, expected), (
                f"[{label}] _propagate{tuple(s)} = {actual}, expected {expected}")
        # passing an explicit `vect` should start from that vector, not the initial one
        custom = np.array([1.0, 0.0, 0.0])
        assert np.allclose(cand._propagate([0], vect=custom),
                           ref._propagate([0], vect=custom)), (
            f"[{label}] _propagate with explicit vect argument is incorrect")
    print("All tests passed!")


def test_sequence_probability(HMM):
    for label, tensor, init in _fixtures():
        ref, cand = solutions.HMM(tensor, init), HMM(tensor, init)
        n_obs = tensor.shape[0]
        for s in _all_seqs(n_obs):
            expected = ref.sequence_probability(list(s))
            actual = cand.sequence_probability(list(s))
            assert np.isclose(actual, expected), (
                f"[{label}] sequence_probability{tuple(s)} = {actual}, expected {expected}")
        # proper distribution: probabilities over all length-L sequences sum to 1
        for length in range(1, 5):
            total = sum(cand.sequence_probability(list(s))
                        for s in product(range(n_obs), repeat=length))
            assert np.isclose(total, 1.0), (
                f"[{label}] sum of P(length-{length} sequences) = {total}, expected 1")
    print("All tests passed!")


def test_conditional_next_token_probabilities(HMM):
    for label, tensor, init in _fixtures():
        ref, cand = solutions.HMM(tensor, init), HMM(tensor, init)
        n_obs = tensor.shape[0]
        for s in _reachable_seqs(ref, n_obs):
            expected = ref.conditional_next_token_probabilities(s)
            actual = cand.conditional_next_token_probabilities(s)
            assert np.allclose(actual, expected), (
                f"[{label}] conditional_next_token_probabilities{tuple(s)} = {actual}, "
                f"expected {expected}")
            assert np.isclose(np.sum(actual), 1.0), (
                f"[{label}] NTP{tuple(s)} sums to {np.sum(actual)}, expected 1")
    # a zero-probability sequence must raise ValueError (Z1R has unreachable sequences)
    tensor, init = processes.z1r(0.5), processes.uniform_initial(3)
    ref, cand = solutions.HMM(tensor, init), HMM(tensor, init)
    zero_seq = _first_zero_prob_seq(ref, 2)
    try:
        cand.conditional_next_token_probabilities(zero_seq)
    except ValueError:
        pass
    else:
        raise AssertionError(
            f"expected ValueError for zero-probability sequence {zero_seq}")
    print("All tests passed!")


def test_all_next_token_probabilities(HMM):
    for label, tensor, init in _fixtures():
        ref, cand = solutions.HMM(tensor, init), HMM(tensor, init)
        expected = ref.all_next_token_probabilities(4)
        actual = cand.all_next_token_probabilities(4)
        assert () in actual, f"[{label}] missing the empty-sequence () entry"
        assert set(actual) == set(expected), (
            f"[{label}] sequence key sets differ: "
            f"extra={set(actual) - set(expected)}, missing={set(expected) - set(actual)}")
        for k in expected:
            assert np.allclose(actual[k], expected[k]), (
                f"[{label}] all_next_token_probabilities[{k}] = {actual[k]}, "
                f"expected {expected[k]}")
    print("All tests passed!")


# ----------------------------------------------------------------------------
# Part 2: belief states
# ----------------------------------------------------------------------------

def test_belief_state(HMM):
    for label, tensor, init in _fixtures():
        ref, cand = solutions.HMM(tensor, init), HMM(tensor, init)
        n_obs = tensor.shape[0]
        for s in _reachable_seqs(ref, n_obs):
            expected = ref.belief_state(s)
            actual = cand.belief_state(s)
            assert np.allclose(actual, expected), (
                f"[{label}] belief_state{tuple(s)} = {actual}, expected {expected}")
            assert np.isclose(np.sum(actual), 1.0), (
                f"[{label}] belief_state{tuple(s)} sums to {np.sum(actual)}, expected 1")
    # a zero-probability sequence must raise ValueError
    tensor, init = processes.z1r(0.5), processes.uniform_initial(3)
    ref, cand = solutions.HMM(tensor, init), HMM(tensor, init)
    zero_seq = _first_zero_prob_seq(ref, 2)
    try:
        cand.belief_state(zero_seq)
    except ValueError:
        pass
    else:
        raise AssertionError(
            f"expected ValueError for zero-probability sequence {zero_seq}")
    print("All tests passed!")


def test_belief_update(HMM):
    for label, tensor, init in _fixtures():
        ref, cand = solutions.HMM(tensor, init), HMM(tensor, init)
        n_obs = tensor.shape[0]
        for s in _reachable_seqs(ref, n_obs, max_len=3):
            belief = ref.belief_state(s)
            reachable = ref.ntp_from_belief_state(belief)
            for obs in range(n_obs):
                if reachable[obs] <= 1e-12:
                    continue
                expected = ref.belief_update(belief, obs)
                actual = cand.belief_update(belief, obs)
                assert np.allclose(actual, expected), (
                    f"[{label}] belief_update(belief_state{tuple(s)}, {obs}) = {actual}, "
                    f"expected {expected}")
                assert np.isclose(np.sum(actual), 1.0), (
                    f"[{label}] belief_update result sums to {np.sum(actual)}, expected 1")
    # an impossible observation from a belief state must raise ValueError (Z1R)
    tensor, init = processes.z1r(0.5), processes.uniform_initial(3)
    ref, cand = solutions.HMM(tensor, init), HMM(tensor, init)
    found = False
    for s in _reachable_seqs(ref, 2, max_len=3):
        belief = ref.belief_state(s)
        reachable = ref.ntp_from_belief_state(belief)
        for obs in range(2):
            if reachable[obs] <= 1e-12:
                try:
                    cand.belief_update(belief, obs)
                except ValueError:
                    found = True
                else:
                    raise AssertionError(
                        f"expected ValueError for impossible observation {obs} "
                        f"from belief_state{tuple(s)}")
                break
        if found:
            break
    assert found, "could not find an impossible (belief, obs) pair to exercise the guard"
    print("All tests passed!")


def test_ntp_from_belief_state(HMM):
    for label, tensor, init in _fixtures():
        ref, cand = solutions.HMM(tensor, init), HMM(tensor, init)
        n_obs = tensor.shape[0]
        for s in _reachable_seqs(ref, n_obs):
            belief = ref.belief_state(s)
            expected = ref.ntp_from_belief_state(belief)
            actual = cand.ntp_from_belief_state(belief)
            assert np.allclose(actual, expected), (
                f"[{label}] ntp_from_belief_state(belief_state{tuple(s)}) = {actual}, "
                f"expected {expected}")
            assert np.isclose(np.sum(actual), 1.0), (
                f"[{label}] next-token distribution sums to {np.sum(actual)}, expected 1")
    print("All tests passed!")


def test_all_belief_states(HMM):
    for label, tensor, init in _fixtures():
        ref, cand = solutions.HMM(tensor, init), HMM(tensor, init)
        expected = ref.all_belief_states(4)
        actual = cand.all_belief_states(4)
        assert () in actual, f"[{label}] missing the empty-sequence () entry"
        assert set(actual) == set(expected), (
            f"[{label}] sequence key sets differ: "
            f"extra={set(actual) - set(expected)}, missing={set(expected) - set(actual)}")
        for k in expected:
            assert np.allclose(actual[k], expected[k]), (
                f"[{label}] all_belief_states[{k}] = {actual[k]}, expected {expected[k]}")
    print("All tests passed!")

In [ ]:
%%writefile plotting.py
"""Interactive simplex plotting for the HMM exercises.

Self-contained — only numpy + plotly are required; scikit-learn is
imported lazily and only used for the >4-dimensional PCA path (the example
processes here are 3-4 dimensional, so it is never needed).
"""

import numpy as np
import plotly.graph_objects as go


def regular_simplex_vertices(n: int) -> np.ndarray:
    """Vertices of a regular (n-1)-simplex, centered, as an (n, n-1) array.

    The n probability-simplex corners e_i live in the hyperplane sum=1; centering
    them and projecting onto an orthonormal basis of the ones-orthogonal-complement
    yields a regular simplex in R^(n-1) (equilateral triangle for n=3, regular
    tetrahedron for n=4). Orientation is arbitrary -- only used for visualization.
    """
    centered = np.eye(n) - 1.0 / n
    # Rows of `centered` span the (n-1)-dim subspace orthogonal to the all-ones
    # vector; Vt[:n-1] is an orthonormal basis of it.
    _, _, vt = np.linalg.svd(centered)
    return centered @ vt[: n - 1].T


def _coords_to_rgb(beliefs: np.ndarray) -> list:
    """Map each belief vector to an RGB color via its first up-to-3 coordinates.

    For a 3-state model this is the exact barycentric coloring (each simplex
    corner -> pure red/green/blue); for more states only the first 3 coordinates
    drive the color.
    """
    rgb = np.zeros((len(beliefs), 3))
    k = min(3, beliefs.shape[1])
    rgb[:, :k] = np.clip(beliefs[:, :k], 0.0, 1.0)
    return ["rgb(%d,%d,%d)" % tuple((c * 255).astype(int)) for c in rgb]


def plot_belief_states(beliefs, sequences=None, path=None, title=None,
                       color_by_coords=True):
    """Plot HMM belief states on the hidden-state simplex (see plot_simplex_points)."""
    n = np.asarray(beliefs).shape[1]
    if title is None:
        title = f"Belief states ({n} hidden states)"
    return plot_simplex_points(beliefs, sequences, path, title, color_by_coords)


def plot_next_token_distributions(ntps, sequences=None, path=None,
                                  title=None, color_by_coords=True):
    """Plot next-token distributions on the symbol simplex (see plot_simplex_points)."""
    n = np.asarray(ntps).shape[1]
    if title is None:
        title = f"Next-token distributions ({n} symbols)"
    return plot_simplex_points(ntps, sequences, path, title, color_by_coords)


def plot_simplex_points(points, sequences=None, path=None, title=None,
                        color_by_coords=True):
    """Build an interactive Plotly scatter of probability vectors on a simplex.

    points: (N, d) array of probability vectors (complex inputs, e.g. GHMM
        beliefs/NTPs, are cast to their real part).
    sequences: optional length-N list of the observation sequences that induced
        each point; shown on hover. If None, hover shows the point index.
    path: if given, also write the figure to this HTML file. The Plotly figure is
        always returned (renders inline in a notebook).
    color_by_coords: if True, color each point by its coordinates (RGB from the
        first up-to-3 components); for d == 3 this is the exact barycentric coloring.

    The points are projected onto the probability simplex, then:
      - d == 2 -> 1D strip (x-axis)
      - d == 3 -> 2D triangle
      - d == 4 -> 3D tetrahedron
      - d  > 4 -> PCA to the top 3 components, 3D scatter
    """
    beliefs = np.real(np.asarray(points, dtype=complex))
    n_points, n = beliefs.shape
    if n < 2:
        raise ValueError(f"need at least 2 simplex dimensions to plot, got {n}.")
    if title is None:
        title = f"Simplex points ({n}-simplex)"

    if sequences is not None:
        if len(sequences) != n_points:
            raise ValueError(
                f"sequences has length {len(sequences)} but there are {n_points} points."
            )
        hover = [f"input sequence: {tuple(s)}" if len(tuple(s)) else "input sequence: () [start]"
                 for s in sequences]
    else:
        hover = [f"index {i}" for i in range(n_points)]

    coords = beliefs @ regular_simplex_vertices(n)

    marker = dict(size=5, opacity=0.7)
    if color_by_coords:
        marker["color"] = _coords_to_rgb(beliefs)
    hovertemplate = "%{text}<extra></extra>"

    if n == 2:
        fig = go.Figure(
            go.Scatter(
                x=coords[:, 0], y=np.zeros(n_points), mode="markers",
                marker=marker, text=hover, hovertemplate=hovertemplate,
            )
        )
        fig.update_yaxes(visible=False)

    elif n == 3:
        verts = regular_simplex_vertices(3)
        loop = np.vstack([verts, verts[0]])  # close the triangle
        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=loop[:, 0], y=loop[:, 1], mode="lines",
            line=dict(color="black"), hoverinfo="skip", showlegend=False,
        ))
        fig.add_trace(go.Scatter(
            x=coords[:, 0], y=coords[:, 1], mode="markers",
            marker=marker, text=hover, hovertemplate=hovertemplate, showlegend=False,
        ))
        fig.update_yaxes(scaleanchor="x", scaleratio=1, autorange="reversed")

    elif n == 4:
        verts = regular_simplex_vertices(4)
        edges = [(i, j) for i in range(4) for j in range(i + 1, 4)]
        ex, ey, ez = [], [], []
        for i, j in edges:
            ex += [verts[i, 0], verts[j, 0], None]
            ey += [verts[i, 1], verts[j, 1], None]
            ez += [verts[i, 2], verts[j, 2], None]
        fig = go.Figure()
        fig.add_trace(go.Scatter3d(
            x=ex, y=ey, z=ez, mode="lines",
            line=dict(color="black"), hoverinfo="skip", showlegend=False,
        ))
        fig.add_trace(go.Scatter3d(
            x=coords[:, 0], y=coords[:, 1], z=coords[:, 2], mode="markers",
            marker=marker, text=hover, hovertemplate=hovertemplate, showlegend=False,
        ))
        fig.update_scenes(yaxis_autorange="reversed")

    else:  # n > 4: PCA to 3D
        from sklearn.decomposition import PCA

        coords = PCA(n_components=3).fit_transform(coords)
        fig = go.Figure(go.Scatter3d(
            x=coords[:, 0], y=coords[:, 1], z=coords[:, 2], mode="markers",
            marker=marker, text=hover, hovertemplate=hovertemplate,
        ))
        fig.update_scenes(
            xaxis_title="PC1", yaxis_title="PC2", zaxis_title="PC3"
        )

    fig.update_layout(title=title)
    if path is not None:
        fig.write_html(path)
    return fig

In [ ]:
# Best-effort auto-reload of the helper modules written above (silently skipped
# where IPython's autoreload extension is unavailable, e.g. some Colab 3.12 images).
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception:
    pass

import numpy as np
from collections.abc import Sequence

import processes
import solutions
import tests

# Shape aliases (documentation only).
type TransitionTensor = np.ndarray  # (n_obs, n_states, n_states)
type StateVector = np.ndarray       # (n_states,) -- belief / propagated state vector
type TokenDist = np.ndarray         # (n_obs,)    -- distribution over next tokens

---

## Exercises

The cells above are **setup**. First run the cell below to define the `HMM` class, then work through the exercises in order: for each, complete the `# YOUR CODE HERE` cell (it attaches the method to `HMM` via `HMM.<name> = <name>`), then run its `tests.test_<name>(HMM)` cell — it prints **All tests passed!** when your implementation is correct.

The `HMM` skeleton — each exercise adds a method to this class via `HMM.<name> = <name>`.

In [ ]:
# The non-belief-state methods from Part 1 are GIVEN here so this notebook is
# standalone. You implement only the belief-state methods below.

class HMM:
    """HMM defined by an observation-indexed transition tensor and an initial vector.

    Shapes
    ------
    transition_tensor        : (n_obs, n_states, n_states)
    belief / state vectors   : (n_states,)
    next-token distributions : (n_obs,)
    """

    def __init__(self, transition_tensor: TransitionTensor, initial_vect: StateVector) -> None:
        self.transition_tensor = np.asarray(transition_tensor)
        self.initial_vect = np.atleast_1d(np.asarray(initial_vect)).ravel()

    def validate(self) -> None:
        """Assert that transition_tensor and initial_vect define a valid HMM.

        The observation-summed state-transition matrix must be row-stochastic and
        the initial vector must be a probability distribution. Raises AssertionError
        otherwise.
        """
        assert (self.transition_tensor >= 0).all(), "transition_tensor must be non-negative"
        transition_matrix = self.transition_tensor.sum(axis=0)
        assert np.allclose(transition_matrix.sum(axis=1), 1.0), \
            "summed transition matrix must be row-stochastic (each row sums to 1)"
        assert (self.initial_vect >= 0).all(), "initial_vect must be non-negative"
        assert np.isclose(self.initial_vect.sum(), 1.0), "initial_vect must sum to 1"

    def _propagate(self, sequence: Sequence[int], vect: StateVector | None = None) -> StateVector:
        """Run a sequence through the tensor, returning the unnormalized state vector."""
        vect = self.initial_vect if vect is None else vect
        for obs in sequence:
            vect = np.einsum("i,ij->j", vect, self.transition_tensor[obs])
        return vect

    def sequence_probability(self, sequence: Sequence[int]) -> float:
        return np.sum(self._propagate(sequence))

    def conditional_next_token_probabilities(self, sequence: Sequence[int]) -> TokenDist:
        seq = list(sequence)
        p_seq = self.sequence_probability(seq)
        if np.allclose(p_seq, 0.0):
            raise ValueError(f"sequence {tuple(sequence)} has zero probability; "
                             "next-token distribution is undefined.")
        n_obs = self.transition_tensor.shape[0]
        joint = np.array([self.sequence_probability(seq + [k]) for k in range(n_obs)])
        return joint / p_seq

    def all_next_token_probabilities(self, max_depth: int) -> dict[tuple[int, ...], TokenDist]:
        """Next-token distribution for every reachable sequence of length 0..max_depth.

        Built breadth-first; a continuation is reachable iff its probability in the
        current distribution is nonzero (zero-probability branches are pruned).
        Uses conditional_next_token_probabilities directly -- no belief-state machinery.
        Includes () -> the prior next-token distribution.
        """
        prior = self.conditional_next_token_probabilities(())
        result = {(): prior}
        frontier = [((), prior)]
        for _ in range(max_depth):
            next_frontier = []
            for seq, ntp in frontier:
                for obs in range(len(ntp)):
                    if np.allclose(ntp[obs], 0.0):
                        continue  # zero-probability continuation -> prune
                    new_seq = seq + (obs,)
                    child = self.conditional_next_token_probabilities(new_seq)
                    result[new_seq] = child
                    next_frontier.append((new_seq, child))
            frontier = next_frontier
        return result

### Exercise - implement `belief_state`

> **Difficulty:** 🔴🔴⚪⚪⚪  
> **Importance:** 🔵🔵🔵🔵⚪  
> 
> You should spend up to ~10 minutes on this exercise.

The belief state is the observer's posterior over the current hidden state given everything seen so far: $\eta^{(x_{1:L})}_j = P(\text{state} = j \mid x_{1:L})$, a vector of shape `(n_states,)` that sums to 1. Where `_propagate` returns the *unnormalized* forward vector $\alpha^{(x_{1:L})}$, with $\alpha^{(x_{1:L})}_j = P(x_{1:L},\ \text{state} = j)$, the belief state is just $\alpha$ rescaled to a probability distribution.

By the definition of conditional probability,
$$
\eta^{(x_{1:L})}_j = \frac{P(x_{1:L},\ \text{state}=j)}{P(x_{1:L})} = \frac{\alpha^{(x_{1:L})}_j}{\sum_k \alpha^{(x_{1:L})}_k}.
$$
The normalizer $\sum_k \alpha^{(x_{1:L})}_k$ is exactly `sequence_probability(sequence)`. If it is zero the sequence is impossible under the model, so no posterior exists — raise `ValueError` in that case rather than dividing by zero.

This normalization is what makes belief states a finite-memory sufficient statistic: the scalar magnitude of $\alpha^{(x_{1:L})}$ (the running joint probability) is discarded, and the resulting unit vector carries all the information from $x_{1:L}$ needed to predict the future.

<details><summary>Hint</summary>

Call `self._propagate(sequence)` to get $\alpha^{(x_{1:L})}$, take its sum, and return $\alpha^{(x_{1:L})}$ divided by that sum. Guard the sum against zero (e.g. `np.allclose(norm, 0.0)`) before dividing and raise `ValueError` for impossible sequences.

</details>

In [ ]:
def belief_state(self, sequence: Sequence[int]) -> StateVector:
    # YOUR CODE HERE
    raise NotImplementedError()

HMM.belief_state = belief_state

In [ ]:
tests.test_belief_state(HMM)

<details>
<summary>Solution</summary>

```python
def belief_state(self, sequence: Sequence[int]) -> StateVector:
    vect = self._propagate(sequence)
    norm = np.sum(vect)
    if np.allclose(norm, 0.0):
        raise ValueError(f"sequence {tuple(sequence)} has zero probability; belief state is undefined.")
    return vect / norm

HMM.belief_state = belief_state
```
</details>

### Exercise - implement `belief_update`

> **Difficulty:** 🔴🔴⚪⚪⚪  
> **Importance:** 🔵🔵🔵⚪⚪  
> 
> You should spend up to ~10-15 minutes on this exercise.

Instead of calculating the belief state for every sequence, it is convenient to maintain a belief and update it after each new observation. Given the current belief `belief_state` (shape `(n_states,)`) and a single observation `obs`, return the updated belief over the *next* hidden state.

The belief updating expression is given by:

$$
\eta^{(x_{1:t})} \mapsto \eta^{(x_{1:t+1})} = \frac{\eta^{(x_{1:t})}T^{(x_{t+1})}}{\eta^{(x_{1:t})}T^{(x_{t+1})}\mathbf{1}}
$$

If the normalizer is $0$, the observation is impossible under the current belief and there is no valid posterior — raise a `ValueError` in that case rather than dividing by zero.

<details><summary>Hint</summary>

Compute the unnormalized update with a single contraction over the source state:

```python
np.einsum("i,ij->j", belief_state, self.transition_tensor[obs])
```

Then check whether the result sums to (near) zero — raise `ValueError` if so — otherwise divide by its sum so the returned vector sums to 1.

</details>

In [ ]:
def belief_update(self, belief_state: StateVector, obs: int) -> StateVector:
    # YOUR CODE HERE
    raise NotImplementedError()

HMM.belief_update = belief_update

In [ ]:
tests.test_belief_update(HMM)

<details>
<summary>Solution</summary>

```python
def belief_update(self, belief_state: StateVector, obs: int) -> StateVector:
    update = np.einsum("i,ij->j", belief_state, self.transition_tensor[obs])
    norm = np.sum(update)
    if np.allclose(norm, 0.0):
        raise ValueError(f"observation {obs} has zero probability from this belief state; cannot update.")
    return update / norm

HMM.belief_update = belief_update
```
</details>

### Exercise - implement `ntp_from_belief_state`

> **Difficulty:** 🔴🔴🔴⚪⚪  
> **Importance:** 🔵🔵🔵⚪⚪  
> 
> You should spend up to ~10-15 minutes on this exercise.

A belief state $\eta^{(x_{1:t})} \in \mathbb{R}^{n_\text{states}}$ summarizes everything the process knows about the latent state given the tokens seen so far: $\eta^{(x_{1:t})}_i = P(\text{state} = i \mid x_{1:t})$, with $\sum_i \eta^{(x_{1:t})}_i = 1$. This method maps that belief directly to a distribution over the next emitted symbol, with no reference to any particular sequence.

The observation-indexed transition tensor $T$ has shape $(n_\text{obs}, n_\text{states}, n_\text{states})$, where $T[k, i, j] = P(\text{emit } k,\ j \mid i)$. Marginalizing the next-state index $j$ gives the conditional probability of emitting $k$ given being in state $i$. Weighting by the belief and summing over the current state $i$ yields

$$
P(\text{next} = k \mid x_{1:t}) = \eta^{(x_{1:t})} T[k] \mathbf{1} =  \sum_{i, j} \eta^{(x_{1:t})}_i \, T[k, i, j].
$$

Because $\eta^{(x_{1:t})}$ sums to $1$ and $T$ is properly normalized (summing $T$ over $k$ and $j$ gives a row-stochastic state-transition matrix), the resulting length-$n_\text{obs}$ vector is already a valid probability distribution — no renormalization needed.

<details><summary>Hint</summary>

Contract the belief against the current-state axis of $T$ and sum out the next-state axis in one shot:

```python
np.einsum("i,kij->k", belief_state, self.transition_tensor)
```

The output is indexed by $k$ (the symbol), giving the token distribution directly.

</details>

In [ ]:
def ntp_from_belief_state(self, belief_state: StateVector) -> TokenDist:
    # YOUR CODE HERE
    raise NotImplementedError()

HMM.ntp_from_belief_state = ntp_from_belief_state

In [ ]:
tests.test_ntp_from_belief_state(HMM)

<details>
<summary>Solution</summary>

```python
def ntp_from_belief_state(self, belief_state: StateVector) -> TokenDist:
    return np.einsum("i,kij->k", belief_state, self.transition_tensor)

HMM.ntp_from_belief_state = ntp_from_belief_state
```
</details>

### Exercise - implement `all_belief_states`

> **Difficulty:** 🔴🔴🔴⚪⚪  
> **Importance:** 🔵🔵⚪⚪⚪  
> 
> You should spend up to ~15 minutes on this exercise.

This method enumerates the belief state for every reachable observation sequence of length $0$ through `max_depth`, returning a dict keyed by the sequence tuple. The empty key `()` maps to the prior belief `belief_state(())`, and each key $(x_1, \dots, x_k)$ maps to the posterior over hidden states given that the process has emitted exactly that sequence.

This is the **mixed-state presentation** of the process: rather than tracking the hidden Markov state, we track the observer's belief (a distribution over states) and how it evolves as observations arrive. Each emitted symbol drives a Bayesian update $b \mapsto \text{belief\_update}(b, x)$, so the set of reachable beliefs forms the points of the mixed-state space. For processes like Mess3 this set is uncountable in the limit, and its finite-depth approximation traces out a fractal in the probability simplex.

Build the dict via breadth-first search. Start from `belief_state(())` keyed by `()`. To expand a node $(x_1,\dots,x_k)$ at depth $<$ `max_depth`, try each observation $x$: the continuation is reachable iff $P(x \mid b) > 0$, which `belief_update` signals by *not* raising. Zero-probability continuations are pruned, so the keys of the returned dict are exactly the sequences the process can actually produce.

<details><summary>Hint</summary>

No einsum here. Do a BFS over sequence tuples: maintain a frontier of `(seq, belief)` pairs, seed it with `((), belief_state(()))`, and for each frontier node at depth less than `max_depth` loop over all observations, calling `belief_update`. Wrap that call in `try/except ValueError` — a raise means the continuation has zero probability, so skip it; otherwise record `seq + (x,)` with its updated belief and push it onto the next frontier.

</details>

In [ ]:
def all_belief_states(self, max_depth: int) -> dict[tuple[int, ...], StateVector]:
    # YOUR CODE HERE
    raise NotImplementedError()

HMM.all_belief_states = all_belief_states

In [ ]:
tests.test_all_belief_states(HMM)

<details>
<summary>Solution</summary>

```python
def all_belief_states(self, max_depth: int) -> dict[tuple[int, ...], StateVector]:
    """Belief state for every reachable sequence of length 0..max_depth.

    Returns a dict mapping each observation sequence (as a tuple) to its
    belief state, starting from the empty sequence () -> prior belief.
    Built breadth-first via belief_update; zero-probability continuations
    are pruned (they have no defined belief state).
    """
    n_obs = self.transition_tensor.shape[0]
    prior = self.belief_state(())
    result = {(): prior}
    frontier = [((), prior)]
    for _ in range(max_depth):
        next_frontier = []
        for seq, belief in frontier:
            for obs in range(n_obs):
                try:
                    updated = self.belief_update(belief, obs)
                except ValueError:
                    continue  # zero-probability continuation -> prune
                new_seq = seq + (obs,)
                result[new_seq] = updated
                next_frontier.append((new_seq, updated))
        frontier = next_frontier
    return result

HMM.all_belief_states = all_belief_states
```
</details>

## Demo

With the belief-state methods implemented, we can collect the reachable belief states and view their geometry interactively (via the local `plotting.py`). Belief states live on the **hidden-state** simplex: **Mess3** (3 states) gives a 2D triangle whose reachable beliefs trace out a fractal, and **arch** (4 states) gives a 3D tetrahedron. Hover a point to see the inducing sequence.

In [ ]:
import plotting

hmm = HMM(processes.mess3(0.15, 0.2), processes.uniform_initial(3))

beliefs = hmm.all_belief_states(5)
print("# distinct reachable sequences <= depth 5:", len(beliefs))
print("belief after observing (0, 1, 2):", np.round(hmm.belief_state((0, 1, 2)), 4))
print("next-token dist from that belief :", np.round(hmm.ntp_from_belief_state(hmm.belief_state((0, 1, 2))), 4))

# Mess3 (3 hidden states): belief states live on the 2-simplex (a triangle) and
# trace out a fractal.
depth = 7
mess3_beliefs = hmm.all_belief_states(depth)
plotting.plot_belief_states(
    np.array([np.real(v) for v in mess3_beliefs.values()]),
    sequences=list(mess3_beliefs.keys()),
    title=f"Mess3 belief-state geometry (depth {depth})",
)

In [ ]:
# The arch process: 4 hidden states, 3 symbols. Belief states lie on the 4-state
# simplex, so they are plotted in 3D (a tetrahedron).
import plotting

arch_hmm = HMM(processes.arch(0.6), processes.uniform_initial(4))
depth = 7
arch_beliefs = arch_hmm.all_belief_states(depth)
plotting.plot_belief_states(
    np.array([np.real(v) for v in arch_beliefs.values()]),
    sequences=list(arch_beliefs.keys()),
    title=f"Arch belief-state geometry (depth {depth})",
)

### Bring your own process

The processes above come from `processes.py`, but nothing is special about them — **any** non-negative transition tensor whose observation-summed matrix is row-stochastic defines a valid HMM. Here we define one **inline** (not in `processes.py`): the **Fern** process (3 states, 2 symbols). `validate()` confirms it is well-formed, then it runs through exactly the same machinery — and its 3-state belief geometry traces out its own fractal in the 2-simplex.

In [ ]:
def fern(x: float) -> np.ndarray:
    """Fern process (custom): 3 hidden states, 2 symbols."""
    assert 0.0 <= x <= 1.0
    return np.array([
        [[0.3942, 0.00512, 0.0381], [0.0, 0.53, 0.0], [0.0, 0.326 * x, 0.554]],
        [[0.3358, 0.01088, 0.2159], [0.0, 0.0, 0.47], [0.12, 0.326 * (1 - x), 0.0]],
    ])

fern_hmm = HMM(fern(0.5), processes.uniform_initial(3))
fern_hmm.validate()  # a custom process still has to be a valid HMM

fern_depth = 12
fern_beliefs = fern_hmm.all_belief_states(fern_depth)
plotting.plot_belief_states(
    np.array([np.real(v) for v in fern_beliefs.values()]),
    sequences=list(fern_beliefs.keys()),
    title=f"Fern belief-state geometry (depth {fern_depth})",
)